In [23]:
import pandas as pd
import numpy as np 

In [24]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from catboost import CatBoostClassifier

In [25]:
pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [26]:
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    fbeta_score, f1_score, accuracy_score
)
from sklearn.base import clone

In [33]:
test=pd.read_csv("C:/Users/Administrator/Desktop/채원이/test_data_fin.csv", encoding="utf-8")
train=pd.read_csv("C:/Users/Administrator/Desktop/채원이/train_data_fin.csv",  encoding="utf-8")

In [36]:
#dummy함수 정의
import pandas as pd

def preprocess_train_test(
    train: pd.DataFrame,
    test: pd.DataFrame,
    dummy_cols=("학년", "전임교원_확보율", "입학세부구분"),
    drop_cols=("혁신수업", "연구재단등재지", "Unnamed: 0","계열"),
    ):
    train = train.copy()
    test = test.copy()

    # 제외 변수 drop (train/test 둘 다에 없을 수도 있으니 errors="ignore")
    train = train.drop(columns=list(drop_cols), errors="ignore")
    test  = test.drop(columns=list(drop_cols), errors="ignore")

    # 더미 처리 대상 중 실제로 존재하는 컬럼만
    dummy_cols = [c for c in dummy_cols if (c in train.columns or c in test.columns)]

    # train+test를 합쳐서 한 번에 더미 만들고 다시 분리 (컬럼 불일치 방지)
    n_train = len(train)
    both = pd.concat([train, test], axis=0, ignore_index=True)

    # 더미 처리, 범주형 -> 숫자
    both = pd.get_dummies(both, columns=dummy_cols, drop_first=False)

    X_train = both.iloc[:n_train, :].reset_index(drop=True)
    X_test  = both.iloc[n_train:, :].reset_index(drop=True)

    # (선택) 혹시 남아있는 object가 있으면 확인
    # print(X_train.dtypes[X_train.dtypes == "object"])

    return X_train, X_test

In [37]:
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    fbeta_score, f1_score, accuracy_score
)
from sklearn.base import clone

BETA = 2.0
N_SPLITS = 5
RANDOM_STATE = 42

# 타깃 변수명
target_col = "학적상태"

# 1) y 분리
y_train_before = train[target_col].copy()
y_test  = test[target_col].copy()

# 2) X 분리
X_train_raw = train.drop(columns=[target_col])
X_test_raw  = test.drop(columns=[target_col])

# 3) X 전처리
X_train_before, X_test = preprocess_train_test(
    X_train_raw,
    X_test_raw,
    dummy_cols=("학년", "전임교원_확보율", "입학세부구분"),
    drop_cols=("혁신수업", "연구재단등재지", "Unnamed: 0","계열"),
)


In [38]:
from imblearn.over_sampling import SMOTE

# SMOTE 객체 생성
smote = SMOTE(random_state=RANDOM_STATE)

# SMOTE 적용 (train 데이터에만)
X_train, y_train = smote.fit_resample(X_train_before, y_train_before)

# 확인
print("Before SMOTE:")
print(y_train_before.value_counts())
print("\nAfter SMOTE:")
print(y_train.value_counts())

Before SMOTE:
학적상태
0    13964
1      284
Name: count, dtype: int64

After SMOTE:
학적상태
0    13964
1    13964
Name: count, dtype: int64


In [39]:
print(X_train.value_counts())

일반휴학학기수  평점평균      취업률  평균취득학점이상  학년_1  학년_2   학년_3   학년_4   전임교원_확보율_-1  전임교원_확보율_0  전임교원_확보율_1  입학세부구분_0  입학세부구분_1  입학세부구분_2
0        0.000000  49   0         True  False  False  False  True         False       False       False     True      True        68
                   53   0         True  False  False  False  True         False       False       False     True      False       58
                   47   0         True  False  False  False  True         False       False       False     True      False       45
                   40   0         True  False  False  False  True         False       True        True      True      False       39
                   63   0         True  False  False  False  True         False       False       False     True      True        38
                                                                                                                                  ..
                   42   0         True  False  False  False  False        F

# Catboost

In [11]:
import sys
!"{sys.executable}" -m pip install catboost

In [40]:
from catboost import CatBoostClassifier

def fit_catboost_predict_proba(X_tr, y_tr, X_va, y_va, params):
    model = CatBoostClassifier(
        **params,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE,
        verbose=0,
        thread_count=-1
    )

    model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
    proba = model.predict_proba(X_va)[:, 1]
    return model, proba


# SVM

In [41]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

def make_svm_model(X_example, random_state=42):
    # 더미화가 끝났으니 전체 컬럼을 numeric으로 보고 스케일링만
    num_cols = X_example.columns.tolist()

    preprocess = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_cols),
        ],
        remainder="drop"
    )

    return Pipeline([
        ("prep", preprocess),
        ("clf", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            probability=False,
            random_state=random_state
        ))
    ])

m = make_svm_model(X_train, random_state=RANDOM_STATE)
m.fit(X_train, y_train)

scores = m.decision_function(X_test)

print("score min / mean / max:", scores.min(), scores.mean(), scores.max())
print("positives predicted (score>=0):", (scores >= 0).sum(), "/", len(scores))
print("true positive rate:", y_test.mean())


score min / mean / max: -3.1327198022926446 -1.464530246697058 1.387899269035489
positives predicted (score>=0): 416 / 6106
true positive rate: 0.020962987225679658


# RF

In [42]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

def make_rf_model(X_example, random_state=RANDOM_STATE):
    return Pipeline([
        ("clf", RandomForestClassifier(
            random_state=random_state,
            n_jobs=-1
        ))
    ])


## gridsearch (train으로 -> valid)

In [43]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.metrics import roc_auc_score, average_precision_score

N_SPLITS = 5
RANDOM_STATE = 42

def custom_search_Xy(X, y, model_name, make_model_fn=None, param_grid=None):
    if not param_grid:
        raise ValueError("param_grid must be provided")

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    rows = []

    for params_raw in ParameterGrid(param_grid):
        fold_roc, fold_pr = [], []

        for tr_idx, va_idx in skf.split(X, y):
            X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
            X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

            if model_name == "catboost":
                m, score = fit_catboost_predict_proba(X_tr, y_tr, X_va, y_va, params_raw)
            else:
                if make_model_fn is None:
                    raise ValueError("make_model_fn must be provided for non-catboost models")

                m = make_model_fn(X_tr)

                params_fit = dict(params_raw)      # fit용 복사본
                params_fit.pop("class_weights", None)
                params_fit.pop("scale_pos_weight", None)

                m.set_params(**params_fit)
                m.fit(X_tr, y_tr)

                # SVM(probability=False)면 decision_function이 score
                if hasattr(m, "predict_proba"):
                    score = m.predict_proba(X_va)[:, 1]
                else:
                    score = m.decision_function(X_va)

            fold_roc.append(roc_auc_score(y_va, score))
            fold_pr.append(average_precision_score(y_va, score))

        rows.append({
            "model": model_name,
            "params": dict(params_raw),  # ✅ 원본 그대로 저장
            "mean_ROC-AUC": float(np.mean(fold_roc)),
            "mean_PR-AUC": float(np.mean(fold_pr)),
        })

    res = (pd.DataFrame(rows)
           .sort_values(["mean_ROC-AUC", "mean_PR-AUC"], ascending=False)
           .reset_index(drop=True))
    best = res.iloc[0].to_dict()
    return best, res




### parameter 후보

In [21]:
## full grid search, no balanced
svm_grid = {
    "clf__kernel": ["rbf"],
    "clf__C": [0.25, 1.0, 4.0, 16.0, 64.0],
    "clf__gamma": ["scale", 0.001, 0.01, 0.1, 1.0],
}

rf_grid = {
    "clf__n_estimators": [300, 600],
    "clf__max_depth": [None, 10, 20, 40],
    "clf__min_samples_leaf": [1, 3, 5],
    "clf__max_features": ["sqrt", 0.5],
}

cat_grid = {
    "iterations": [800, 1200],
    "learning_rate": [0.03, 0.05, 0.1],
    "depth": [4, 6, 8],
    "l2_leaf_reg": [1, 3, 10],
}

## no balanced best parameters search

In [22]:
best_svm, res_svm = custom_search_Xy(
    X_train, y_train,
    model_name="svm",
    make_model_fn=make_svm_model,
    param_grid=svm_grid
)

best_rf, res_rf = custom_search_Xy(
    X_train, y_train,
    model_name="rf",
    make_model_fn=make_rf_model,
    param_grid=rf_grid
)  

best_cat, res_cat = custom_search_Xy(
    X_train, y_train,
    model_name="catboost",
    param_grid=cat_grid
)

best_svm, best_rf, best_cat


({'model': 'svm',
  'params': {'clf__C': 64.0, 'clf__gamma': 'scale', 'clf__kernel': 'rbf'},
  'mean_ROC-AUC': 0.9897116846991814,
  'mean_PR-AUC': 0.9895124744696877},
 {'model': 'rf',
  'params': {'clf__max_depth': 20,
   'clf__max_features': 'sqrt',
   'clf__min_samples_leaf': 1,
   'clf__n_estimators': 600},
  'mean_ROC-AUC': 0.9970042889922368,
  'mean_PR-AUC': 0.996778901252584},
 {'model': 'catboost',
  'params': {'depth': 8,
   'iterations': 1200,
   'l2_leaf_reg': 1,
   'learning_rate': 0.1},
  'mean_ROC-AUC': 0.997146500760296,
  'mean_PR-AUC': 0.9971675006420181})

In [44]:
best_svm = {
    "model": "svm",
    "params": {
        "clf__C": 64.0,
        "clf__gamma": "scale",
        "clf__kernel": "rbf",
    },
    "mean_ROC-AUC": 0.9897116846991814,
    "mean_PR-AUC": 0.9895124744696877,
}

best_rf = {
    "model": "rf",
    "params": {
        "clf__max_depth": 20,
        "clf__max_features": "sqrt",
        "clf__min_samples_leaf": 1,
        "clf__n_estimators": 600,
    },
    "mean_ROC-AUC": 0.9970042889922368,
    "mean_PR-AUC": 0.996778901252584,
}

best_cat = {
    "model": "catboost",
    "params": {
        "depth": 8,
        "iterations": 1200,
        "l2_leaf_reg": 1,
        "learning_rate": 0.1,
    },
    "mean_ROC-AUC": 0.997146500760296,
    "mean_PR-AUC": 0.9971675006420181,
}


## Best params 지표 (SMOTE)

In [46]:
from sklearn.metrics import roc_auc_score, f1_score, recall_score, accuracy_score, precision_score

def metrics_simple(y_true, score, thr):
    pred = (score >= thr).astype(int)
    return {
        "ROC-AUC": roc_auc_score(y_true, score),
        "Recall": recall_score(y_true, pred),
        "F1": f1_score(y_true, pred),
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Pred_Pos": int(pred.sum()),
        "Thr_used": thr,
    }

target_col = "학적상태"
y_test  = test[target_col].copy()

def fit_final_and_eval(model_name, best):
    if model_name == "catboost":
        m = CatBoostClassifier(
            **best["params"],
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=RANDOM_STATE,
            verbose=0,
            thread_count=-1
        )
        m.fit(X_train, y_train)
        score = m.predict_proba(X_test)[:, 1]
        thr = 0.5

    elif model_name == "svm":
        m = make_svm_model(X_train)
        m.set_params(**best["params"])
        m.fit(X_train, y_train)
        score = m.decision_function(X_test)
        thr = 0.0

    elif model_name == "rf":
        m = make_rf_model(X_train)     # ✅ 수정 포인트
        m.set_params(**best["params"])
        m.fit(X_train, y_train)
        score = m.predict_proba(X_test)[:, 1]
        thr = 0.5

    mets = metrics_simple(y_test, score, thr)
    return {"model": model_name, **mets}




,model,ROC-AUC,Recall,F1,Accuracy,Precision,Pred_Pos,Thr_used
0,catboost,0.873466,0.250000,0.212625,0.961186,0.184971,173,0.5
1,svm,0.815425,0.414062,0.210736,0.934982,0.141333,375,0.0
2,rf,0.882738,0.234375,0.189873,0.958074,0.159574,188,0.5


In [47]:
summary = (
    pd.DataFrame([
        fit_final_and_eval("svm", best_svm),
        fit_final_and_eval("rf",  best_rf),
        fit_final_and_eval("catboost", best_cat),
    ])
    .sort_values(by="F1", ascending=False)
    .reset_index(drop=True)
)

summary_rounded = summary.round(4)
summary_rounded

,model,ROC-AUC,Recall,F1,Accuracy,Precision,Pred_Pos,Thr_used
0,catboost,0.8735,0.2500,0.2126,0.9612,0.1850,173,0.5
1,svm,0.8154,0.4141,0.2107,0.9350,0.1413,375,0.0
2,rf,0.8827,0.2344,0.1899,0.9581,0.1596,188,0.5
